# NB05: first-round availability sensitivity

- **Purpose:** test how draft-slot associations change when every slot is restricted to rosters whose actual first-round selection remained available for at least 75% of that league's fantasy regular season.
- **Pipeline:** retained team panel -> actual Sleeper first-round selections -> Sleeper-to-GSIS identifier link -> nflverse weekly participation -> availability-conditioned inference.
- **Inputs:** `data/processed/analysis_panel.csv`, retained Sleeper caches, and compact local caches created on the first run.
- **Outputs:** availability linkage, slot summaries, clustered intervals, evaluation metadata, and four Plotly HTML charts under `artifacts/`.
- **Run:** execute after NB04. The first run may fetch missing historic draft-pick endpoints and nflverse participation data; later runs use local caches.
- **Locked definition:** availability equals active NFL weeks before the league's playoff start divided by fantasy regular-season weeks. The threshold is 0.75 for every slot. Missing player links are excluded, not classified as unavailable.
- **Interpretation boundary:** the original injury-inclusive analysis remains primary. This post-draft conditioning is a mechanism sensitivity and does not estimate a causal injury-free draft-slot effect.

| Gate | What it checks | Pass condition |
| ---: | --- | --- |
| 1 | Sleeper pick coverage | Complete first-round picks for at least 3,600 of 3,641 drafts |
| 2 | Player linkage | At least 90% linked to weekly participation in every slot |
| 3 | Conditioned sample size | At least 2,000 available team-seasons in every slot |
| 4 | Cluster inference | Complete league-season resampling with 2,000 draws |
| 5 | Artifact completeness | Four data files and four Plotly HTML charts written |

### What this cell does

- Loads the balanced historical panel and locks the availability threshold, bootstrap count, and seed.
- Verifies the 3,641-league and 43,692-team structure before adding post-draft data.

In [1]:
# CELL [1 load-panel-and-lock-design]
from pathlib import Path
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed
import csv
import hashlib
import json
import math
import re
import time
import unicodedata
from urllib.error import HTTPError, URLError
from urllib.request import Request, urlopen

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
CACHE = ROOT / "data/cache"
PROCESSED = ROOT / "data/processed"
ARTIFACTS = ROOT / "artifacts"
AVAILABILITY_THRESHOLD = 0.75
BOOTSTRAP_REPLICATES = 2_000
SEED = 20260813

panel = pd.read_csv(PROCESSED / "analysis_panel.csv", dtype={"league_id": str, "draft_id": str, "roster_id": int})
league_sizes = panel.groupby(["league_id", "season"]).size()
assert len(panel) == 43_692 and len(league_sizes) == 3_641
assert league_sizes.eq(12).all()
assert panel.groupby("draft_slot").size().eq(3_641).all()

print({
    "league_seasons": int(len(league_sizes)),
    "team_seasons": int(len(panel)),
    "availability_threshold": AVAILABILITY_THRESHOLD,
    "bootstrap_replicates": BOOTSTRAP_REPLICATES,
})

{'league_seasons': 3641, 'team_seasons': 43692, 'availability_threshold': 0.75, 'bootstrap_replicates': 2000}


### Interpreting the output

- The panel contains 3,641 complete league-seasons and 43,692 team-seasons.
- The availability threshold is locked at 75% of each league's fantasy regular season, with 2,000 clustered bootstrap draws.
- Passing preserves the same primary population used by NB01 through NB04.
- It does not validate later player linkage or availability measurement.


### What this cell does

- Builds a compact cache of actual first-round Sleeper picks for all retained drafts.
- Extracts the 2,628 current-era responses from the retained 1.4 GB cache and fetches only missing historic responses.
- Saves each league's playoff-start week so player availability uses the matching fantasy regular-season window.
- Skips source-cache parsing and network calls after the compact caches are complete.

In [2]:
# CELL [2 build-compact-sleeper-caches]
PICK_CACHE_PATH = CACHE / "first_round_pick_cache.jsonl"
LEAGUE_SETTINGS_PATH = CACHE / "league_regular_season_settings.csv"
SOURCE_CACHE_PATH = CACHE / "sleeper_http_cache.json"
HISTORIC_LEAGUES_PATH = CACHE / "historic_user_leagues.jsonl"

needed_drafts = set(panel["draft_id"].astype(str))
needed_leagues = set(panel["league_id"].astype(str))
draft_records = {}
if PICK_CACHE_PATH.exists():
    with PICK_CACHE_PATH.open(encoding="utf-8") as handle:
        for line in handle:
            record = json.loads(line)
            draft_records[record["draft_id"]] = record

league_settings = {}
if LEAGUE_SETTINGS_PATH.exists():
    saved_settings = pd.read_csv(LEAGUE_SETTINGS_PATH, dtype={"league_id": str})
    league_settings = saved_settings.set_index("league_id").to_dict("index")

missing_drafts = needed_drafts - set(draft_records)
missing_leagues = needed_leagues - set(league_settings)
extracted_from_source = 0
if missing_drafts or missing_leagues:
    try:
        import ijson
    except ImportError as error:
        raise ImportError("Install ijson to stream the retained Sleeper cache on the first run") from error
    source_records = []
    with SOURCE_CACHE_PATH.open("rb") as handle:
        for key, payload in ijson.kvitems(handle, ""):
            parts = key.split("/")
            if len(parts) == 3 and parts[0] == "draft" and parts[2] == "picks" and parts[1] in missing_drafts:
                first_round = [pick for pick in (payload or []) if int(pick.get("round") or 0) == 1]
                record = {"draft_id": parts[1], "source": "retained_sleeper_http_cache", "picks": first_round}
                draft_records[parts[1]] = record
                source_records.append(record)
                extracted_from_source += 1
            elif len(parts) == 2 and parts[0] == "league" and parts[1] in missing_leagues:
                settings = payload.get("settings") or {}
                league_settings[parts[1]] = {"season": int(payload.get("season")), "playoff_week_start": settings.get("playoff_week_start"), "source": "retained_sleeper_http_cache"}
    if source_records:
        with PICK_CACHE_PATH.open("a", encoding="utf-8") as handle:
            for record in source_records:
                handle.write(json.dumps(record) + chr(10))

if missing_leagues:
    with HISTORIC_LEAGUES_PATH.open(encoding="utf-8") as handle:
        for line in handle:
            for league in json.loads(line).get("payload") or []:
                league_id = str(league.get("league_id"))
                if league_id in missing_leagues:
                    settings = league.get("settings") or {}
                    league_settings[league_id] = {"season": int(league.get("season")), "playoff_week_start": settings.get("playoff_week_start"), "source": "historic_user_leagues_cache"}

missing_drafts = sorted(needed_drafts - set(draft_records))
def fetch_first_round(draft_id):
    url = f"https://api.sleeper.app/v1/draft/{draft_id}/picks"
    request = Request(url, headers={"User-Agent": "fantasy-draft-order-study/0.2"})
    for attempt in range(5):
        try:
            with urlopen(request, timeout=30) as response:
                payload = json.loads(response.read().decode("utf-8"))
            return {"draft_id": draft_id, "source": "sleeper_api_historic_backfill", "picks": [pick for pick in (payload or []) if int(pick.get("round") or 0) == 1]}
        except HTTPError as error:
            if error.code not in {429, 500, 502, 503, 504} or attempt == 4:
                raise
        except (TimeoutError, URLError):
            if attempt == 4:
                raise
        time.sleep(2 ** attempt)

fetched_records = []
if missing_drafts:
    with ThreadPoolExecutor(max_workers=8) as executor:
        futures = {executor.submit(fetch_first_round, draft_id): draft_id for draft_id in missing_drafts}
        for future in as_completed(futures):
            fetched_records.append(future.result())
    fetched_records.sort(key=lambda record: record["draft_id"])
    with PICK_CACHE_PATH.open("a", encoding="utf-8") as handle:
        for record in fetched_records:
            handle.write(json.dumps(record) + chr(10))
            draft_records[record["draft_id"]] = record

league_rows = []
league_seasons = panel[["league_id", "season"]].drop_duplicates()
for row in league_seasons.itertuples(index=False):
    record = league_settings.get(row.league_id, {})
    raw_start = record.get("playoff_week_start")
    default_start = 14 if row.season <= 2020 else 15
    playoff_start = int(raw_start) if raw_start and 2 <= int(raw_start) <= 18 else default_start
    league_rows.append({"league_id": row.league_id, "season": int(row.season), "playoff_week_start": playoff_start, "regular_season_weeks": playoff_start - 1, "source": record.get("source", "era_default")})
league_settings_frame = pd.DataFrame(league_rows)
league_settings_frame.to_csv(LEAGUE_SETTINGS_PATH, index=False)

assert set(draft_records) == needed_drafts
assert len(league_settings_frame) == len(league_sizes)
usable_drafts = {draft_id: record for draft_id, record in draft_records.items() if len(record.get("picks") or []) == 12}
empty_pick_drafts = sorted(needed_drafts - set(usable_drafts))
assert len(usable_drafts) >= 3600
print({
    "drafts_required": len(needed_drafts),
    "drafts_cached": len(draft_records),
    "drafts_with_complete_first_round": len(usable_drafts),
    "drafts_without_recoverable_picks": len(empty_pick_drafts),
    "extracted_from_retained_cache": extracted_from_source,
    "historic_endpoints_fetched_this_run": len(fetched_records),
    "league_settings_rows": len(league_settings_frame),
})

{'drafts_required': 3641, 'drafts_cached': 3641, 'drafts_with_complete_first_round': 3626, 'drafts_without_recoverable_picks': 15, 'extracted_from_retained_cache': 0, 'historic_endpoints_fetched_this_run': 0, 'league_settings_rows': 3641}


### Interpreting the output

- Complete first-round picks were recovered for 3,626 of 3,641 drafts. Fifteen historic drafts returned empty Sleeper pick payloads and are excluded from NB05 only.
- Later reruns use the compact pick and league-settings caches rather than the 1.4 GB retained source cache.
- The recoverable-pick subset is large enough for within-slot availability comparisons.
- Cache completeness does not establish correct player linkage or injury status.


### What this cell does

- Joins each roster to its actual first-round Sleeper selection and player identifiers.
- Builds a weekly NFL participation panel for 2018 through 2025 and converts active weeks into fantasy-regular-season availability.
- Marks team-seasons as available when the first-round player appears in at least 75% of the league's regular-season weeks.

In [3]:
# CELL [3 link-players-and-measure-availability]
import nflreadpy as nfl

def fold_name(value):
    text = unicodedata.normalize("NFKD", str(value or ""))
    text = "".join(character for character in text if not unicodedata.combining(character))
    return re.sub(r"[^a-z0-9]", "", text.lower())

analysis_panel = panel.loc[panel["draft_id"].isin(usable_drafts)].copy()
assert analysis_panel["draft_id"].nunique() == len(usable_drafts)

pick_rows = []
for draft_id, record in usable_drafts.items():
    for pick in record["picks"]:
        metadata = pick.get("metadata") or {}
        pick_rows.append({
            "draft_id": draft_id,
            "draft_slot": int(pick["draft_slot"]),
            "roster_id": int(pick["roster_id"]),
            "pick_no": int(pick["pick_no"]),
            "sleeper_player_id": str(pick.get("player_id") or metadata.get("player_id") or ""),
            "player_name": f"{metadata.get('first_name', '')} {metadata.get('last_name', '')}".strip(),
            "player_position": metadata.get("position"),
            "player_team": metadata.get("team"),
        })
picks = pd.DataFrame(pick_rows)
assert picks.groupby("draft_id").size().eq(12).all()
assert picks.groupby(["draft_id", "draft_slot"]).size().eq(1).all()

linked = analysis_panel.merge(picks.drop(columns=["roster_id"]), on=["draft_id", "draft_slot"], how="left", validate="one_to_one")
linked = linked.merge(league_settings_frame[["league_id", "playoff_week_start", "regular_season_weeks"]], on="league_id", how="left", validate="many_to_one")
assert linked["sleeper_player_id"].fillna("").ne("").all()

ids = nfl.load_ff_playerids().to_pandas()
ids = ids.loc[ids["sleeper_id"].notna() & ids["gsis_id"].notna(), ["sleeper_id", "gsis_id", "name", "position"]].copy()
ids["sleeper_id"] = ids["sleeper_id"].astype(int).astype(str)
ids["gsis_id"] = ids["gsis_id"].astype(str)
ids = ids.drop_duplicates("sleeper_id", keep="first")
ids["fold_name"] = ids["name"].map(fold_name)
linked = linked.merge(ids[["sleeper_id", "gsis_id", "name"]].rename(columns={"sleeper_id": "sleeper_player_id", "name": "crosswalk_name"}), on="sleeper_player_id", how="left", validate="many_to_one")

weekly_path = CACHE / "nflverse_weekly_participation_2018_2025.csv"
if weekly_path.exists():
    weekly = pd.read_csv(weekly_path)
else:
    weekly = nfl.load_player_stats(list(range(2018, 2026))).to_pandas()
    weekly = weekly.loc[weekly["season_type"].eq("REG"), ["player_id", "player_display_name", "position", "season", "week"]].copy()
    weekly.to_csv(weekly_path, index=False)

weekly["player_id"] = weekly["player_id"].astype(str)
weekly["season"] = weekly["season"].astype(int)
weekly["week"] = weekly["week"].astype(int)
weekly["fold_name"] = weekly["player_display_name"].map(fold_name)

fallback = (
    weekly.sort_values(["season", "week"])
    .drop_duplicates(["season", "fold_name", "position"], keep="first")
    [["season", "fold_name", "position", "player_id"]]
    .rename(columns={"player_id": "gsis_id_fallback"})
)
linked["fold_name"] = linked["player_name"].map(fold_name)
linked = linked.merge(
    fallback,
    left_on=["season", "fold_name", "player_position"],
    right_on=["season", "fold_name", "position"],
    how="left",
)
linked["gsis_id"] = linked["gsis_id"].fillna(linked["gsis_id_fallback"])
linked = linked.drop(columns=["position", "gsis_id_fallback"], errors="ignore")

active_counts = (
    weekly.merge(linked[["gsis_id", "season", "regular_season_weeks"]].dropna().drop_duplicates(), left_on=["player_id", "season"], right_on=["gsis_id", "season"], how="inner")
)
active_counts = active_counts.loc[active_counts["week"] < (active_counts["regular_season_weeks"] + 1)]
active_counts = active_counts.groupby(["gsis_id", "season", "regular_season_weeks"]).size().rename("active_weeks").reset_index()
linked = linked.merge(active_counts, on=["gsis_id", "season", "regular_season_weeks"], how="left")
linked["active_weeks"] = linked["active_weeks"].fillna(0).astype(int)
linked["player_linked"] = linked["gsis_id"].notna()
linked["availability_rate"] = np.where(linked["player_linked"], linked["active_weeks"] / linked["regular_season_weeks"], np.nan)
linked["first_round_available"] = pd.Series(pd.NA, index=linked.index, dtype="boolean")
linked.loc[linked["player_linked"], "first_round_available"] = linked.loc[linked["player_linked"], "availability_rate"] >= AVAILABILITY_THRESHOLD

print({
    "first_round_rows": int(len(linked)),
    "linked_share": round(float(linked["player_linked"].mean()), 4),
    "available_share_among_linked": round(float(linked.loc[linked["player_linked"], "first_round_available"].mean()), 4),
    "mean_regular_season_weeks": round(float(linked["regular_season_weeks"].mean()), 2),
})

{'first_round_rows': 43512, 'linked_share': 0.9988, 'available_share_among_linked': 0.7137, 'mean_regular_season_weeks': 13.85}


### Interpreting the output

- Among recoverable drafts, 99.88% of first-round selections linked to weekly NFL participation.
- About 71% of linked first-round selections met the 75% availability threshold.
- Slot 1 had the highest unavailability rate at 42.3%, versus about 26% for slots 3 through 8.
- The measure uses weekly participation, not a verified injury-status label.


### What these tests guard

- Requires at least 90% player linkage in every draft slot.
- Requires at least 2,000 available linked team-seasons in every draft slot.
- Builds full-sample and availability-conditioned slot summaries for points, top six, and top scorer.

In [4]:
# CELL [4 validate-coverage-and-summarize]
def summarize_slots(frame, sample_label):
    rows = []
    for slot, group in frame.groupby("draft_slot"):
        rows.append({
            "sample": sample_label,
            "draft_slot": int(slot),
            "n_team_seasons": int(len(group)),
            "mean_points_zscore": float(group["points_zscore"].mean()),
            "top_6_rate": float(group["top_6_points"].mean()),
            "top_scorer_rate": float(group["top_regular_season_scorer"].mean()),
            "unavailable_share": float((~group["first_round_available"].fillna(False)).mean()) if "first_round_available" in group else np.nan,
        })
    return pd.DataFrame(rows).sort_values("draft_slot")

coverage = (
    linked.groupby("draft_slot")
    .agg(n=("league_id", "size"), linked_share=("player_linked", "mean"), available_n=("first_round_available", lambda series: int(series.fillna(False).sum())))
    .reset_index()
)
assert (coverage["linked_share"] >= 0.90).all()
assert (coverage["available_n"] >= 2000).all()

available = linked.loc[linked["first_round_available"] == True].copy()
summary = pd.concat([
    summarize_slots(linked.assign(first_round_available=linked["player_linked"] & linked["first_round_available"].fillna(False)), "Full retained panel"),
    summarize_slots(available, "First-round available"),
], ignore_index=True)

slot_1_full = summary.loc[(summary["sample"] == "Full retained panel") & (summary["draft_slot"] == 1)].iloc[0]
slot_4_full = summary.loc[(summary["sample"] == "Full retained panel") & (summary["draft_slot"] == 4)].iloc[0]
slot_1_available = summary.loc[(summary["sample"] == "First-round available") & (summary["draft_slot"] == 1)].iloc[0]
slot_4_available = summary.loc[(summary["sample"] == "First-round available") & (summary["draft_slot"] == 4)].iloc[0]
print({
    "min_linked_share": round(float(coverage["linked_share"].min()), 4),
    "min_available_n": int(coverage["available_n"].min()),
    "available_team_seasons": int(len(available)),
    "slot_1_top6_full": round(float(slot_1_full["top_6_rate"]), 4),
    "slot_1_top6_available": round(float(slot_1_available["top_6_rate"]), 4),
    "slot_4_minus_slot_1_top6_full_pp": round(100 * (slot_4_full["top_6_rate"] - slot_1_full["top_6_rate"]), 2),
    "slot_4_minus_slot_1_top6_available_pp": round(100 * (slot_4_available["top_6_rate"] - slot_1_available["top_6_rate"]), 2),
})

{'min_linked_share': 0.997, 'min_available_n': 2091, 'available_team_seasons': 31018, 'slot_1_top6_full': 0.4606, 'slot_1_top6_available': 0.5328, 'slot_4_minus_slot_1_top6_full_pp': np.float64(6.8), 'slot_4_minus_slot_1_top6_available_pp': np.float64(2.84)}


### Reading the test result

- Minimum linkage by slot exceeded 90%, and every slot retained at least 2,091 available team-seasons.
- Slot 1 top-six rate rises from 46.1% in the pick-recoverable panel to 53.3% when first-round selections remain available.
- The slot 4 minus slot 1 top-six gap shrinks from 6.80 percentage points to 2.84 points under availability conditioning.
- Conditioning on post-draft availability is not a causal estimate of drafting without injury risk.


### What this cell does

- Estimates league-clustered intervals for top-six rates in the full panel and the availability-conditioned sample.
- Compares slot 1 and slot 4 under the same 2,000-draw resampling design used in NB03.

In [5]:
# CELL [5 clustered-bootstrap-comparison]
def slot_top6_means(blocks, slots=range(1, 13)):
    totals = {slot: 0.0 for slot in slots}
    counts = {slot: 0 for slot in slots}
    for block in blocks:
        for row in block:
            slot = row["draft_slot"]
            totals[slot] += row["top_6_points"]
            counts[slot] += 1
    return {slot: (totals[slot] / counts[slot] if counts[slot] else np.nan) for slot in slots}

def bootstrap_effects(frame, sample_label):
    blocks = [group.to_dict("records") for _, group in frame.groupby(["league_id", "season"])]
    observed = slot_top6_means(blocks)
    rng = np.random.default_rng(SEED)
    draws = {slot: [] for slot in range(1, 13)}
    for _ in range(BOOTSTRAP_REPLICATES):
        sample = [blocks[int(index)] for index in rng.integers(0, len(blocks), size=len(blocks))]
        estimate = slot_top6_means(sample)
        for slot in range(1, 13):
            draws[slot].append(estimate[slot] - 0.5)
    rows = []
    for slot in range(1, 13):
        values = np.array(draws[slot], dtype=float)
        values = values[~np.isnan(values)]
        rows.append({
            "sample": sample_label,
            "draft_slot": slot,
            "estimate_minus_baseline": observed[slot] - 0.5,
            "ci_95_low": float(np.quantile(values, 0.025)),
            "ci_95_high": float(np.quantile(values, 0.975)),
            "n_league_seasons": len(blocks),
            "n_team_seasons": int(sum(len(block) for block in blocks)),
        })
    return pd.DataFrame(rows)

full_effects = bootstrap_effects(linked, "Full retained panel")
available_effects = bootstrap_effects(available, "First-round available")
effects = pd.concat([full_effects, available_effects], ignore_index=True)

def contrast(effects_frame, sample_label):
    subset = effects_frame.loc[effects_frame["sample"] == sample_label].set_index("draft_slot")
    return float(subset.loc[4, "estimate_minus_baseline"] - subset.loc[1, "estimate_minus_baseline"])

print({
    "full_slot_1_effect_pp": round(100 * float(full_effects.loc[full_effects["draft_slot"] == 1, "estimate_minus_baseline"].iloc[0]), 2),
    "full_slot_4_effect_pp": round(100 * float(full_effects.loc[full_effects["draft_slot"] == 4, "estimate_minus_baseline"].iloc[0]), 2),
    "available_slot_1_effect_pp": round(100 * float(available_effects.loc[available_effects["draft_slot"] == 1, "estimate_minus_baseline"].iloc[0]), 2),
    "available_slot_4_effect_pp": round(100 * float(available_effects.loc[available_effects["draft_slot"] == 4, "estimate_minus_baseline"].iloc[0]), 2),
    "full_slot4_minus_slot1_pp": round(100 * contrast(effects, "Full retained panel"), 2),
    "available_slot4_minus_slot1_pp": round(100 * contrast(effects, "First-round available"), 2),
})

{'full_slot_1_effect_pp': -3.94, 'full_slot_4_effect_pp': 2.85, 'available_slot_1_effect_pp': 3.28, 'available_slot_4_effect_pp': 6.12, 'full_slot4_minus_slot1_pp': 6.8, 'available_slot4_minus_slot1_pp': 2.84}


### Interpreting the output

- In the pick-recoverable panel, slot 1 is 3.94 points below the 50% baseline and slot 4 is 2.85 points above it.
- After availability conditioning, slot 1 moves to 3.28 points above baseline and slot 4 to 6.12 points above baseline.
- The slot 4 advantage shrinks but does not disappear.
- Surviving gaps do not prove that injury caused the original association.


### What this cell does

- Plots unavailable first-round rates by draft slot.
- Compares top-six rates before and after availability conditioning.
- Shows league-cluster intervals for both samples.
- Plots the slot 4 minus slot 1 top-six gap under both definitions.

In [6]:
# CELL [6 plot-availability-sensitivity]
unavailable_by_slot = (
    linked.assign(unavailable=~linked["first_round_available"].fillna(False))
    .groupby("draft_slot")["unavailable"]
    .mean()
    .reset_index()
)
fig_unavailable = px.bar(unavailable_by_slot, x="draft_slot", y="unavailable", title="Share of first-round selections below 75% availability")
fig_unavailable.update_layout(template="plotly_white", height=480, xaxis_title="Draft slot", yaxis_title="Unavailable share", showlegend=False)
fig_unavailable.update_yaxes(tickformat=".0%")

rate_plot = summary[["sample", "draft_slot", "top_6_rate"]].copy()
fig_rates = px.line(rate_plot, x="draft_slot", y="top_6_rate", color="sample", markers=True, title="Top-six rate before and after availability conditioning")
fig_rates.update_layout(template="plotly_white", height=500, xaxis_title="Draft slot", yaxis_title="Top-six rate", legend_title="Sample")
fig_rates.update_yaxes(tickformat=".0%")
fig_rates.add_hline(y=0.5, line_dash="dash", line_color="#d93025")

interval_plot = effects.copy()
fig_intervals = go.Figure()
for sample, color in [("Full retained panel", "#3367d6"), ("First-round available", "#188038")]:
    subset = interval_plot.loc[interval_plot["sample"] == sample].sort_values("draft_slot")
    fig_intervals.add_trace(go.Scatter(
        x=subset["estimate_minus_baseline"],
        y=subset["draft_slot"],
        mode="markers",
        name=sample,
        marker={"color": color, "size": 9},
        error_x={"type": "data", "symmetric": False, "array": subset["ci_95_high"] - subset["estimate_minus_baseline"], "arrayminus": subset["estimate_minus_baseline"] - subset["ci_95_low"]},
        hovertemplate="Sample %{fullData.name}<br>Draft slot %{y}<br>Top-six effect %{x:.2%}<extra></extra>",
    ))
fig_intervals.add_vline(x=0, line_dash="dash", line_color="#d93025")
fig_intervals.update_layout(title="Clustered top-six effects by draft slot", template="plotly_white", height=620, xaxis_title="Top-six rate minus 50%", yaxis_title="Draft slot", yaxis={"autorange": "reversed", "dtick": 1})
fig_intervals.update_xaxes(tickformat=".0%")

gap_rows = pd.DataFrame([
    {"sample": "Full retained panel", "slot4_minus_slot1": contrast(effects, "Full retained panel")},
    {"sample": "First-round available", "slot4_minus_slot1": contrast(effects, "First-round available")},
])
fig_gap = px.bar(gap_rows, x="sample", y="slot4_minus_slot1", title="Slot 4 minus slot 1 top-six gap")
fig_gap.update_layout(template="plotly_white", height=460, xaxis_title="Sample", yaxis_title="Top-six gap", showlegend=False)
fig_gap.update_yaxes(tickformat=".0%")

figures = {
    "availability_01_unavailable_by_slot.html": fig_unavailable,
    "availability_02_top_six_rates.html": fig_rates,
    "availability_03_clustered_intervals.html": fig_intervals,
    "availability_04_slot4_minus_slot1.html": fig_gap,
}
for filename, figure in figures.items():
    figure.write_html(ARTIFACTS / filename, include_plotlyjs="cdn")
    figure.show()

print({"plotly_charts_written": len(figures)})

{'plotly_charts_written': 4}


### Interpreting the output

- Slot 1 is the most exposed to first-round unavailability. Conditioning raises every slot's top-six rate because absences are removed symmetrically, but the relative gain is largest at slot 1.
- The slot 4 minus slot 1 gap falls from 6.80 to 2.84 percentage points.
- Clustered intervals still place both slots above the 50% baseline after conditioning, with slot 4 remaining ahead.
- Charts do not prove causality or that unavailable weeks were exclusively injury weeks.


### What this cell does

- Writes the availability-linked panel, sample summaries, clustered intervals, and evaluation metadata.
- Displays the before-versus-after slot summary with a polished or interactive table when available.

In [7]:
# CELL [7 save-artifacts-and-display]
linked_path = PROCESSED / "first_round_availability_panel.csv"
summary_path = ARTIFACTS / "availability_slot_summary.csv"
effects_path = ARTIFACTS / "availability_slot_effects.csv"
evaluation_path = ARTIFACTS / "availability_evaluation.json"

export_columns = [
    "league_id", "season", "draft_id", "draft_slot", "roster_id", "pick_no", "sleeper_player_id",
    "player_name", "player_position", "gsis_id", "player_linked", "regular_season_weeks",
    "active_weeks", "availability_rate", "first_round_available", "points_zscore", "top_6_points",
    "top_regular_season_scorer",
]
linked[export_columns].to_csv(linked_path, index=False)
summary.to_csv(summary_path, index=False)
effects.to_csv(effects_path, index=False)

evaluation = {
    "historical_league_seasons": int(len(league_sizes)),
    "historical_team_seasons": int(len(panel)),
    "pick_recoverable_league_seasons": int(len(usable_drafts)),
    "drafts_without_recoverable_picks": int(len(empty_pick_drafts)),
    "availability_threshold": AVAILABILITY_THRESHOLD,
    "linked_share": float(linked["player_linked"].mean()),
    "available_team_seasons": int(len(available)),
    "min_available_n_per_slot": int(coverage["available_n"].min()),
    "slot_1_top6_full": float(slot_1_full["top_6_rate"]),
    "slot_1_top6_available": float(slot_1_available["top_6_rate"]),
    "slot_4_top6_full": float(slot_4_full["top_6_rate"]),
    "slot_4_top6_available": float(slot_4_available["top_6_rate"]),
    "slot4_minus_slot1_top6_full": float(slot_4_full["top_6_rate"] - slot_1_full["top_6_rate"]),
    "slot4_minus_slot1_top6_available": float(slot_4_available["top_6_rate"] - slot_1_available["top_6_rate"]),
    "primary_result_remains_injury_inclusive": True,
    "plotly_artifacts": list(figures),
}
evaluation_path.write_text(json.dumps(evaluation, indent=2), encoding="utf-8")

display_frame = summary.pivot(index="draft_slot", columns="sample", values=["n_team_seasons", "mean_points_zscore", "top_6_rate"]).sort_index()
display_frame.columns = [f"{metric} | {sample}" for metric, sample in display_frame.columns]
display_frame = display_frame.reset_index()
try:
    from great_tables import GT
    display(GT(display_frame).fmt_number(columns=[column for column in display_frame.columns if "zscore" in column], decimals=3).fmt_percent(columns=[column for column in display_frame.columns if "top_6_rate" in column], decimals=1))
except ImportError:
    try:
        from itables import init_notebook_mode, show
        init_notebook_mode(all_interactive=True)
        show(display_frame)
    except ImportError:
        display(display_frame)

print({
    "files_written": 4 + len(figures),
    "slot4_minus_slot1_full_pp": round(100 * evaluation["slot4_minus_slot1_top6_full"], 2),
    "slot4_minus_slot1_available_pp": round(100 * evaluation["slot4_minus_slot1_top6_available"], 2),
})

draft_slot,n_team_seasons | First-round available,n_team_seasons | Full retained panel,mean_points_zscore | First-round available,mean_points_zscore | Full retained panel,top_6_rate | First-round available,top_6_rate | Full retained panel
1,2091.0,3626.0,0.076,−0.103,53.3%,46.1%
2,2597.0,3626.0,0.106,−0.005,54.0%,49.8%
3,2702.0,3626.0,0.138,0.040,55.6%,51.5%
4,2680.0,3626.0,0.135,0.057,56.1%,52.9%
5,2669.0,3626.0,0.126,0.029,54.9%,51.3%
6,2685.0,3626.0,0.109,0.032,53.9%,50.9%
7,2667.0,3626.0,0.080,0.014,54.0%,51.3%
8,2666.0,3626.0,0.106,0.024,53.4%,50.3%
9,2573.0,3626.0,0.059,−0.017,52.1%,49.2%
10,2546.0,3626.0,0.033,−0.045,51.5%,48.7%


{'files_written': 8, 'slot4_minus_slot1_full_pp': 6.8, 'slot4_minus_slot1_available_pp': 2.84}


### Interpreting the output

- Eight artifacts were written: the linked panel, two summary tables, evaluation metadata, and four Plotly charts.
- The before-versus-after gap is 6.80 percentage points in the pick-recoverable sample and 2.84 points after availability conditioning.
- Every first-round selection, availability rate, and outcome remains inspectable outside the notebook.
- These files support mechanism analysis and do not replace the injury-inclusive primary study.


## Conclusion

- First-round unavailability explains a large share of slot 1's weakness: its top-six rate rises from 46.1% to 53.3% when the actual first-round selection remains available for at least 75% of the fantasy regular season.
- Slot 4 still leads after that conditioning, but the slot 4 minus slot 1 gap shrinks from 6.80 to 2.84 percentage points.
- Keep the injury-inclusive analysis as the primary real-world result. Treat NB05 as a mechanism sensitivity based on actual Sleeper first-round picks.
- The notebook does not settle causal draft-order effects, separate injury from other absences, or replace the primary study.
